# 10 · Monitoring et alerting du modèle en production
Dérive des features et des scores (PSI) par rapport à la baseline du champion, performance réelle
quand les labels arrivent, historisation dans monitoring_metrics, échec volontaire en cas d'alerte.

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import pandas as pd
from pyspark.sql import functions as F
from churn.dbx import get_config, write_delta
from churn.config import DRIFT_FEATURES, KEYS, LABEL
from churn import monitoring as MON

cfg = get_config(dbutils)
dbutils.widgets.dropdown("fail_on_alert", "false", ["true", "false"])

preds = spark.table(cfg.t("attrition_predictions"))
snapshot = preds.agg(F.max("snapshot_date")).first()[0]
current = preds.where(F.col("snapshot_date") == snapshot).toPandas()      # le dernier scoring
baseline = spark.table(cfg.t("monitoring_baseline")).toPandas()            # la référence du champion
model_version = int(current.model_version.iloc[0])
print(f"Scoring surveillé : {snapshot} ({len(current)} clients) | baseline : {len(baseline)} lignes")

In [0]:
# 1) Dérive : PSI de chaque feature et des scores
report = MON.drift_report(baseline, current, DRIFT_FEATURES + ["p_churn"], cfg.psi_warn, cfg.psi_alert)

# 2) Performance réelle : sur les mois scorés dont les labels sont maintenant connus
realized = (preds.join(spark.table(cfg.t("attrition_labels")), KEYS)
            .select("snapshot_date", "model_version", "p_churn", LABEL).toPandas())
perf_rows = []
for (snap, ver), g in realized.groupby(["snapshot_date", "model_version"]):
    if g[LABEL].nunique() == 2:
        perf_rows.append({**MON.performance_check(g[LABEL], g.p_churn, cfg.min_auc),
                          "evaluated_snapshot": snap})
perf = pd.DataFrame(perf_rows)

# 3) Historisation
metrics = pd.concat([report.assign(evaluated_snapshot=snapshot), perf], ignore_index=True)
metrics["snapshot_date"] = snapshot
metrics["model_version"] = model_version
metrics["run_ts"] = pd.Timestamp.utcnow()
metrics["feature"] = metrics["feature"].astype("string")
write_delta(spark, metrics, cfg.t("monitoring_metrics"))

overall = MON.overall_status(metrics)
display(metrics.sort_values("value", ascending=False))
print("STATUT GLOBAL :", overall, "| performance réelle disponible :", len(perf_rows), "mois")

In [0]:
if overall == "ALERT" and dbutils.widgets.get("fail_on_alert") == "true":
    alerts = metrics[metrics.status == "ALERT"][["metric", "value"]].to_dict("records")
    raise RuntimeError(f"Monitoring ALERT sur le scoring du {snapshot} : {alerts}. "
                       "Action : analyser la dérive, puis relancer l'entraînement si la performance baisse.")
print("Pas d'alerte bloquante.")